# ⚙️ Feature Engineering & Representation Learning

Feature engineering is the process of using domain knowledge and mathematical transformations to create informative input features that expose underlying data patterns to machine learning algorithms.

This hands-on module covers:
- **Temporal & Datetime Decomposition**: Converting text timestamps into calendar signals
- **Behavioral Indicators**: Deriving day of week, day name, and weekend indicators
- **Seasonal & Business Intervals**: Extracting fiscal quarters and interval boundaries
- **Feature Hygiene**: Purging redundant raw timestamp columns to avoid estimator errors
- **Cyclical Encoding**: Sine and cosine transformations for periodic cyclical features (Months/Hours)
- **Feature Discretization**: Binning continuous features into ordinal ranges


### 1. Library Imports
Import standard data manipulation and visualization packages: `pandas`, `numpy`, `seaborn`, and `matplotlib.pyplot`.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


### 2. Creating Sample Temporal Dataset
Construct a sample DataFrame containing raw date strings across multiple years and months to demonstrate datetime transformation techniques.


In [ ]:
df = pd.DataFrame({
    'Date': ['2026-10-26', '2025-06-29', '2024-02-12', '2026-09-22', '2023-03-16']
})
df


### 3. Inspecting Column Data Types
Check the current data types. Notice that `Date` is stored as a generic `object` (string), which machine learning models cannot interpret numerically or temporally.


In [ ]:
df.dtypes


### 4. Datetime Conversion (`pd.to_datetime`)
Convert the `Date` column from `object` strings into native `datetime64[ns]` format using `pd.to_datetime()`. This unlocks pandas' vectorized datetime accessor (`.dt`).


In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.dtypes


### 5. Extracting Core Calendar Components: Year, Month, Day
Decompose the datetime objects into separate numeric integer features: `Year`, `Month`, and `Day` using the `.dt` accessor.


In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df


### 6. Extracting Behavioral Patterns: Day of Week & Weekend Flag
Derive calendar behavior indicators:
- `DayOfWeek`: 0 = Monday, 6 = Sunday
- `DayName`: Categorical string label (e.g., Monday, Sunday)
- `IsWeekend`: Binary indicator flag ($1$ if Saturday/Sunday, $0$ otherwise)


In [ ]:
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['DayName'] = df['Date'].dt.day_name()
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df


### 7. Extracting Fiscal Intervals: Quarters & Month Boundaries
Extract seasonal fiscal quarters ($1, 2, 3, 4$) and boundary flags (`is_month_end`) often correlated with sales cycles and financial reporting.


In [ ]:
df['Quarter'] = df['Date'].dt.quarter
df['IsMonthEnd'] = df['Date'].dt.is_month_end.astype(int)
df


### 8. Feature Hygiene: Dropping Raw Date Strings
Drop the original unparsed `Date` column and the text `DayName` column to produce a clean, purely numeric tabular matrix ready for Scikit-Learn estimators.


In [ ]:
df_clean = df.drop(columns=['Date', 'DayName']).copy()
print(f"Final Estimator-Ready Matrix Shape: {df_clean.shape}")
df_clean


### 9. Advanced Feature Engineering: Cyclical Encoding
Calendar units like `Month` ($1$ to $12$) or `Hour` ($0$ to $23$) are inherently cyclical: Month 12 (December) is adjacent to Month 1 (January).
Standard linear features treat them as distant ($|12 - 1| = 11$).
By projecting months onto a unit circle using Sine and Cosine transformations, models preserve natural circular continuity:
$$x_{sin} = \sin\left(\frac{2\pi \times x}{T}\right), \quad x_{cos} = \cos\left(\frac{2\pi \times x}{T}\right)$$


In [ ]:
df_clean['Month_sin'] = np.sin(2 * np.pi * df_clean['Month'] / 12)
df_clean['Month_cos'] = np.cos(2 * np.pi * df_clean['Month'] / 12)

df_clean[['Year', 'Month', 'Month_sin', 'Month_cos']]


### 10. Continuous Feature Discretization (Binning)
Binning converts continuous numerical values into discrete categorical buckets (`pd.cut`), which can help capture non-linear threshold effects.


In [ ]:
df_clean['Month_Phase'] = pd.cut(
    df_clean['Day'],
    bins=[0, 10, 20, 31],
    labels=['Early_Month', 'Mid_Month', 'Late_Month']
)
df_clean[['Day', 'Month_Phase']]


### 11. Summary & Best Practices
1. **Model Usability**: Machine learning models cannot consume raw timestamps; decomposing dates into `Year`, `Month`, `Day`, and cyclical transforms makes temporal trends learnable.
2. **Periodic Signals**: Always apply cyclical Sine/Cosine transformations when cyclical recurrence (e.g. hour of day, day of week, month of year) strongly influences target behavior.
3. **Leakage Prevention**: Compute aggregations, scaling parameters, and target encodings strictly on the training partition before transforming test data.
